In [0]:
%scala
/*dbutils.fs.mount(
  source = "wasbs://greentaxi@enterprisehubstorage.blob.core.windows.net/",
  mountPoint = "/mnt/taxi",
  extraConfigs = Map("fs.azure.account.key.enterprisehubstorage.blob.core.windows.net" -> ""))*/

dfutils.ls('/mnt/taxi')
 ^
On line 5: warning: symbol literal is deprecated; use Symbol("/") instead [quickfixable]
 fs.ls('/mnt/taxi')
 ^
On line 5: warning: symbol literal is deprecated; use Symbol("/") instead [quickfixable]
 dbutils.fs.ls('/mnt/taxi')
 ^
On line 5: warning: symbol literal is deprecated; use Symbol("/") instead [quickfixable]

In [0]:
#dbutils.fs.mount(
#    source="wasbs://greentaxi@enterprisehubstorage.blob.core.windows.net/",
#    mount_point="/mnt/nytaxi",
#    extra_configs= {"fs.azure.account.key.enterprisehubstorage.blob.core.windows.net" : ""})

In [0]:
dbutils.fs.ls("/mnt/nytaxi/2024")

[FileInfo(path='dbfs:/mnt/nytaxi/2024/10/', name='10/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/nytaxi/2024/11/', name='11/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/nytaxi/2024/12/', name='12/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/mnt/nytaxi/2024/9/', name='9/', size=0, modificationTime=0)]

In [0]:
from pyspark.sql.functions import col

df = spark.read.format('csv').option('header', 'true').load('/mnt/indusind2/actual_cost_selectedcol/output-enterprise-data-platform')
df = df.filter("SubscriptionName == 'ENTERPRISE-DATA-PLATFORM-DC-PROD-SUBSCRIPTION'  and MeterCategory == 'Virtual Network' and  MeterSubCategory == 'Private Link' and MeterName == 'Standard Data Processed - Ingress'")
df_converted = df.withColumn("EffectivePrice_float", col("EffectivePrice").cast("float"))
df_converted.printSchema()

df_summed = df_converted.groupBy("Date").sum("EffectivePrice_float")
df_summed.show()


root
 |-- SubscriptionId: string (nullable = true)
 |-- SubscriptionName: string (nullable = true)
 |-- ResourceGroup: string (nullable = true)
 |-- ResourceLocation: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- MeterCategory: string (nullable = true)
 |-- MeterSubCategory: string (nullable = true)
 |-- MeterId: string (nullable = true)
 |-- MeterName: string (nullable = true)
 |-- MeterRegion: string (nullable = true)
 |-- UnitOfMeasure: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- EffectivePrice: string (nullable = true)
 |-- ResourceId: string (nullable = true)
 |-- EffectivePrice_float: float (nullable = true)

+----------+-------------------------+
|      Date|sum(EffectivePrice_float)|
+----------+-------------------------+
|07/10/2025|        4074.077618122101|
|07/18/2025|       3757.5367183685303|
|07/28/2025|        6727.560632228851|
|07/27/2025|        6440.028871536255|
|07/05/2025|   

In [0]:
job_paths={'2025':['/mnt/nytaxi/2025/1/', '/mnt/nytaxi/2025/2/', '/mnt/nytaxi/2025/3/', '/mnt/nytaxi/2025/4/', '/mnt/nytaxi/2025/5/'],
            '2024':['/mnt/nytaxi/2024/9/', '/mnt/nytaxi/2024/10/', '/mnt/nytaxi/2024/11/', '/mnt/nytaxi/2024/12/']}

print(job_paths) 

print(dbutils.widgets.get("input"))

{'2025': ['/mnt/nytaxi/2025/1/', '/mnt/nytaxi/2025/2/', '/mnt/nytaxi/2025/3/', '/mnt/nytaxi/2025/4/', '/mnt/nytaxi/2025/5/'], '2024': ['/mnt/nytaxi/2024/9/', '/mnt/nytaxi/2024/10/', '/mnt/nytaxi/2024/11/', '/mnt/nytaxi/2024/12/']}


---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-5123430119568337>, line 6
      1 job_paths={'2025':['/mnt/nytaxi/2025/1/', '/mnt/nytaxi/2025/2/', '/mnt/nytaxi/2025/3/', '/mnt/nytaxi/2025/4/', '/mnt/nytaxi/2025/5/'],
      2             '2024':['/mnt/nytaxi/2024/9/', '/mnt/nytaxi/2024/10/', '/mnt/nytaxi/2024/11/', '/mnt/nytaxi/2024/12/']}
      4 print(job_paths) 
----> 6 print(dbutils.widgets.get("input"))

File /databricks/python_shell/lib/dbruntime/WidgetHandlerImpl.py:82, in WidgetsHandlerImpl.get(self, name)
     42 def get(self, name: str) -> str:
     43     """ Returns the current value of a widget with the given name.
     44 
     45     :param name: Name of the argument to be accessed
   (...)
     80         ```
     81     """
---> 82     return self._notebookArguments.getArgument(name, self._entry_point.getCurrentBindings())

File /databricks/spark/python/

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS adventureworks_sales COMMENT 'AdventureWorks Sales data';


In [0]:
%sql
GRANT USAGE ON CATALOG adventureworks_sales TO `adventureworks_users`;
GRANT USAGE ON SCHEMA adventureworks_sales.default TO `adventureworks_users`;


In [0]:
%sql
drop catalog adventureworks_sales cascade;

In [0]:
%sql
USE CATALOG adventureworks_sales; -- Ensure the user has proper permissions to access this catalog.

CREATE TABLE IF NOT EXISTS default.department
(
   deptcode   INT,
   deptname  STRING,
   location  STRING
);



In [0]:
%sql
ALTER TABLE default.department 
CHANGE COLUMN deptcode COMMENT 'The department code';

ALTER TABLE default.department 
CHANGE COLUMN deptname COMMENT 'The department name';

ALTER TABLE default.department 
CHANGE COLUMN location COMMENT 'location of the department';

COMMENT ON TABLE default.department 
IS 'The table contains information about various departments within the organization. It includes details such as department codes, names, and their respective locations. This data can be used for organizational analysis, resource allocation, and understanding departmental structures.';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS default.customer
(
	CustomerKey INT COMMENT 'Unique identifier for each customer. Format: INTEGER.',
	Customer_ID STRING  COMMENT 'External or business-facing customer identifier. Format: String. Example values: \'AW00016804\', \'AW00020148\' ',
	Customer STRING  COMMENT 'Name of the customer. Format: String. Example values: \'Casey Pal\', \'Louis Sun\' ',
	City STRING  COMMENT 'City where the customer resides. Format: String. Example values: \'Redwood City\', \'North Sydney\', \'Liverpool\', \'San Diego\' ',
	State_Province STRING  COMMENT 'State or province of the customer’s address. Format: String. Example values: \'Wyoming\', \'Ontario\' ',
	Country_Region STRING  COMMENT 'Country or region of the customer. Format: String. Example values: \'United Kingdom\', \'France\' ',
	Postal_Code STRING  COMMENT 'Postal or ZIP code of the customer’s location. Format: String. Example values: \'80074\', \'GL7 1RY\', \'13441\' '
)
USING DELTA 
COMMENT 'Represents individual customers and their geographic details.';

CREATE TABLE IF NOT EXISTS default.date
(
	DateKey INT COMMENT 'Unique identifier for each date entry. Format: INTEGER.',
	Date STRING  COMMENT 'Actual calendar date. Format: YYYY-MM-DD. Example values: \'2022-02-15\', \'2023-12-31\'',
	Fiscal_Year STRING  COMMENT 'Fiscal year corresponding to the date. Format: String. Example values: \'FY2020\', \'FY2019\', \'FY2018\', \'FY2021\'',
	Fiscal_Quarter STRING  COMMENT 'Fiscal quarter (e.g., Q1, Q2) for the date. Format: String. Example values: \'FY2021 Q1\', \'FY2020 Q3\', \'FY2019 Q4\', \'FY2018 Q2\'',
	Month STRING  COMMENT 'Month name along with the year. Format: String. Example values: \'2019 Feb\', \'2019 Jun\'',
	Full_Date STRING  COMMENT 'Full date value, often used for display or sorting. Format: YYYY-MM-DD. Example values: \'2022-02-15\', \'2023-12-31\'',
	MonthKey STRING  COMMENT 'Unique identifier for the month, useful for grouping. Format: String. Example values: \'201909\', \'201903\', \'201801\''
)
USING DELTA 
COMMENT 'Stores calendar and fiscal date information for reporting and analysis.';

CREATE TABLE IF NOT EXISTS default.product
(
	ProductKey INT COMMENT 'Unique identifier for each product. Format: INTEGER.',
	SKU STRING  COMMENT 'Stock Keeping Unit, used for inventory tracking. Format: String. Example values: \'BK-M18B-42\', \'BK-M38S-42\'',
	Product STRING  COMMENT 'Name of the product. Format: String. Example values: \'Classic Vest, L\', \'Classic Vest, M\', \'HL Crankset\'',
	Standard_Cost FLOAT  COMMENT 'Base cost to produce or acquire the product. Format: Float. Example values: 159.56, 24.0, 1374.30',
	Color STRING  COMMENT 'Color variant of the product. Format: String. Example values: \'Silver/Black\', \'Silver\'',
	List_Price FLOAT  COMMENT 'Retail price of the product. Format: Float. Example values: 159.56, 24.0, 1374.30',
	Model STRING  COMMENT 'Model name. Format: String. Example values: \'Bike Wash\', \'Cycling Cap\', \'Minipump\'',
	Subcategory STRING  COMMENT 'Subcategory classification of the product. Format: String. Example values: \'Brakes\', \'Locks\'',
	Category STRING  COMMENT 'High-level category classification. Format: String. Example values: \'Accessories\', \'Bikes\', \'Clothing\''
)
USING DELTA 
COMMENT 'Contains product catalog details including pricing and categorization.';

CREATE TABLE IF NOT EXISTS default.reseller
(
	ResellerKey INT COMMENT 'Unique identifier for each reseller. Format: INTEGER.',
	Reseller_ID STRING  COMMENT 'External or business-facing reseller identifier. Format: String. ',
	Business_Type STRING  COMMENT 'Type of business. Format: String. Example values: \'Specialty Bike Shop\', \'Value Added Reseller\', \'Warehouse\'',
	Reseller STRING  COMMENT 'Name of the reseller company. Format: String. Example values: \'Action Bicycle Specialists\', \'A Typical Bike Shop\'',
	City STRING  COMMENT 'City where the reseller operates. Format: String. Example values: \'Atlanta\', \'Baldwin Park\'',
	State_Province STRING  COMMENT 'State or province of the reseller’s location. Format: String. Example values: \'Connecticut\', \'Hamburg\'',
	Country_Region STRING  COMMENT 'Country or region of the reseller. Format: String. Example values: \'Germany\', \'United States\'',
	Postal_Code STRING  COMMENT 'Postal or ZIP code of the reseller’s address. Format: String. Example values: \'03106\', \'06512\''
)
USING DELTA 
COMMENT 'Captures information about business resellers.';

CREATE TABLE IF NOT EXISTS default.sales
(
	SalesOrderLineKey INT COMMENT 'Unique identifier for each sales order line item. Format: Integer.',
	ResellerKey INT  COMMENT 'Foreign key linking to the reseller involved in the sale. Format: Integer. ',
	CustomerKey INT  COMMENT 'Foreign key linking to the customer. Format: Integer.',
	ProductKey INT  COMMENT 'Foreign key linking to the product sold. Format: Integer.',
	OrderDateKey INT  COMMENT 'Date key for when the order was placed. Format: Integer.',
	DueDateKey INT  COMMENT 'Date key for when the order is due. Format: Integer.',
	ShipDateKey INT  COMMENT 'Date key for when the order was shipped. Format: Integer.',
	SalesTerritoryKey INT  COMMENT 'Foreign key linking to the sales territory. Format: Integer. ',
	Order_Quantity INT COMMENT 'Number of units ordered.  Format: Integer. Example values: 12, 35, 7',
	Unit_Price FLOAT  COMMENT 'Price per unit of the product. Format: Float. Example values: 159.56, 24.0, 1374.30',
	Extended_Amount FLOAT  COMMENT 'Total amount before discounts. Format: Float. Example values: 159.56, 24.0, 1374.30',
	Unit_Price_Discount_Pct STRING  COMMENT 'Discount percentage applied to unit price. Format: Varchar. ',
	Product_Standard_Cost FLOAT  COMMENT 'Standard cost of the product. Format: Float. Example values: 159.56, 24.0, 1374.30',
	Total_Product_Cost FLOAT  COMMENT 'Total cost for the quantity ordered Format: Float. Example values: 159.56, 24.0, 1374.30',
	Sales_Amount FLOAT  COMMENT 'Final sales amount after discounts. Format: Float. Example values: 159.56, 24.0, 1374.30'
)
USING DELTA 
COMMENT 'Tracks individual sales transactions and financial metrics.';


CREATE TABLE IF NOT EXISTS default.salesorder
(
	Channel STRING COMMENT 'Sales channel. Format: String. Example values: \'Reseller\', \'Internet\'',
	SalesOrderLineKey INT  COMMENT 'Foreign key linking to the sales line item. Format: Integer. ',
	Sales_Order STRING  COMMENT 'Identifier for the sales order. Format: String.',
	Sales_Order_Line STRING  COMMENT 'Identifier for the specific line within the order. Format: String.'
)
USING DELTA 
COMMENT 'Provides mapping between sales orders and their line items.';

CREATE TABLE IF NOT EXISTS default.salesterritory
(
	SalesTerritoryKey INT  COMMENT 'Unique identifier for each sales territory. Format: Integer. ',
	Region STRING COMMENT 'Region name. Format: String. Example values: \'Australia\', \'Canada\', \'Northeast\', \'Southwest\'',
	Country STRING  COMMENT 'Country within the region. Format: String. Example values: \'United Kingdom\', \'United States\'',
	Group STRING  COMMENT 'Grouping or classification. Format: String. Example values: \'Corporate HQ\', \'North America\', \'Pacific\', \'Europe\''
)
USING DELTA 
COMMENT 'Entity that defines geographic sales regions and groupings.';


In [0]:
%sql
use schema default;
SELECT so.Channel, COUNT(DISTINCT so.Sales_Order) AS order_count, SUM(s.Sales_Amount) AS total_sales FROM salesorder AS so JOIN sales AS s ON so.SalesOrderLineKey = s.SalesOrderLineKey GROUP BY so.Channel;

Channel,order_count,total_sales
Internet,27659,2.935867807885647E7
Reseller,3796,8.045059577114427E7


In [0]:
%sql
--SELECT table_schema, table_name, comment FROM system.information_schema.tables WHERE table_catalog = 'adventureworks_sales' and table_schema = 'default';
--SELECT column_name, data_type, is_nullable, comment, table_name FROM system.information_schema.columns WHERE table_catalog = 'adventureworks_sales' and table_schema = 'default';

SELECT column_name, data_type, is_nullable, cls.comment as column_comment, cls.table_schema, cls.table_name, tbl.comment as table_comment FROM system.information_schema.columns cls join system.information_schema.tables tbl on tbl.table_schema==cls.table_schema and tbl.table_catalog==cls.table_catalog  WHERE cls.table_catalog = 'adventureworks_sales' and cls.table_schema = 'default';

SELECT column_name, data_type, is_nullable, cls.comment as column_comment, cls.table_schema, cls.table_name, tbl.comment as table_comment FROM system.information_schema.columns cls join system.information_schema.tables tbl on tbl.table_schema==cls.table_schema and tbl.table_catalog==cls.table_catalog  WHERE cls.table_catalog = 'adventureworks_sales' and cls.table_schema = 'default' and tbl.table_catalog = 'adventureworks_sales' and tbl.table_schema = 'default' limit 20;

--select table_schema_table_name, comment from system.information_schema.tables WHERE table_catalog = 'adventureworks_sales' and table_schema = 'dbo'
--join system.information_schema.columns on 

    --  AND table_schema = 'your_schema'
    --  AND table_name = 'your_table';

column_name,data_type,is_nullable,column_comment,table_schema,table_name,table_comment
CustomerKey,INT,YES,Unique identifier for each customer. Format: INTEGER.,default,customer,null
Customer_ID,STRING,YES,"External or business-facing customer identifier. Format: String. Example values: 'AW00016804', 'AW00020148'",default,customer,null
Customer,STRING,YES,"Name of the customer. Format: String. Example values: 'Casey Pal', 'Louis Sun'",default,customer,null
City,STRING,YES,"City where the customer resides. Format: String. Example values: 'Redwood City', 'North Sydney', 'Liverpool', 'San Diego'",default,customer,null
State_Province,STRING,YES,"State or province of the customer’s address. Format: String. Example values: 'Wyoming', 'Ontario'",default,customer,null
Country_Region,STRING,YES,"Country or region of the customer. Format: String. Example values: 'United Kingdom', 'France'",default,customer,null
Postal_Code,STRING,YES,"Postal or ZIP code of the customer’s location. Format: String. Example values: '80074', 'GL7 1RY', '13441'",default,customer,null
CustomerKey,INT,YES,Unique identifier for each customer. Format: INTEGER.,default,customer,Captures information about business resellers.
Customer_ID,STRING,YES,"External or business-facing customer identifier. Format: String. Example values: 'AW00016804', 'AW00020148'",default,customer,Captures information about business resellers.
Customer,STRING,YES,"Name of the customer. Format: String. Example values: 'Casey Pal', 'Louis Sun'",default,customer,Captures information about business resellers.


column_name,data_type,is_nullable,column_comment,table_schema,table_name,table_comment
CustomerKey,INT,YES,Unique identifier for each customer. Format: INTEGER.,default,customer,null
Customer_ID,STRING,YES,"External or business-facing customer identifier. Format: String. Example values: 'AW00016804', 'AW00020148'",default,customer,null
Customer,STRING,YES,"Name of the customer. Format: String. Example values: 'Casey Pal', 'Louis Sun'",default,customer,null
City,STRING,YES,"City where the customer resides. Format: String. Example values: 'Redwood City', 'North Sydney', 'Liverpool', 'San Diego'",default,customer,null
State_Province,STRING,YES,"State or province of the customer’s address. Format: String. Example values: 'Wyoming', 'Ontario'",default,customer,null
Country_Region,STRING,YES,"Country or region of the customer. Format: String. Example values: 'United Kingdom', 'France'",default,customer,null
Postal_Code,STRING,YES,"Postal or ZIP code of the customer’s location. Format: String. Example values: '80074', 'GL7 1RY', '13441'",default,customer,null
CustomerKey,INT,YES,Unique identifier for each customer. Format: INTEGER.,default,customer,Captures information about business resellers.
Customer_ID,STRING,YES,"External or business-facing customer identifier. Format: String. Example values: 'AW00016804', 'AW00020148'",default,customer,Captures information about business resellers.
Customer,STRING,YES,"Name of the customer. Format: String. Example values: 'Casey Pal', 'Louis Sun'",default,customer,Captures information about business resellers.


In [0]:
%sql
select IS_ACCOUNT_GROUP_MEMBER('admin')
--SHOW GROUPS WITH USER `current_user()`


is_account_group_member(admin)
false


In [0]:
%sql
CREATE FUNCTION IF NOT EXISTS us_filter(region STRING)
RETURN IF(IS_ACCOUNT_GROUP_MEMBER('admin'), true, region='US');

CREATE TABLE IF NOT EXISTS dummy_sales (region STRING, id INT);
ALTER TABLE dummy_sales SET ROW FILTER us_filter ON (region);

insert into dummy_sales values('EMEA', 1);
insert into dummy_sales values('US', 2);
insert into dummy_sales values('Asia', 3);

CREATE FUNCTION ssn_mask(ssn STRING)
  RETURN CASE WHEN is_account_group_member('HumanResourceDept') THEN ssn ELSE '***-**-****' END;

CREATE TABLE users (
  name STRING,
  ssn STRING MASK ssn_mask);

insert into users values('Robert Aragon', '489-36-8350');
insert into users values('Ashley Borden', '514-14-8905');
insert into users values('Thomas Conley', '690-05-5315');


num_affected_rows,num_inserted_rows
1,1


num_affected_rows,num_inserted_rows
1,1


num_affected_rows,num_inserted_rows
1,1


num_affected_rows,num_inserted_rows
1,1


num_affected_rows,num_inserted_rows
1,1


num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
select * from dummy_sales;

select * from users;

In [0]:
%sql
-- Hydrate the data into the default schema from the dbo schema. 
use schema dbo;
show tables;
--DESCRIBE TABLE `sales-territory`;

--insert into default.salesterritory(salesterritorykey, group, country, region) select salesterritorykey, group, country, region from `sales-territory`;
--insert into default.salesorder(salesorderlinekey, channel, sales_order_line, sales_order) select salesorderlinekey, channel, sales_order_line, sales_order from `sales-order`;
--insert into default.sales(SalesOrderLineKey, ResellerKey, CustomerKey, ProductKey, OrderDateKey, DueDateKey, ShipDateKey, SalesTerritoryKey, Order_Quantity, Unit_Price, Extended_Amount, Unit_Price_Discount_Pct, Product_Standard_Cost, Total_Product_Cost, Sales_Amount) select SalesOrderLineKey, ResellerKey, CustomerKey, ProductKey, OrderDateKey, DueDateKey, ShipDateKey, SalesTerritoryKey, Order_Quantity, Unit_Price, Extended_Amount, Unit_Price_Discount_Pct, Product_Standard_Cost, Total_Product_Cost, Sales_Amount from dbo.sales;
--insert into default.reseller(ResellerKey, Reseller_ID, Business_Type, Reseller, City, State_Province, Country_Region, Postal_Code) select ResellerKey, Reseller_ID, Business_Type, Reseller, City, State_Province, Country_Region, Postal_Code from dbo.reseller;
--insert into default.product(ProductKey, SKU, Product, Standard_Cost, Color, List_Price, Model, Subcategory, Category) select ProductKey, SKU, Product, Standard_Cost, Color, List_Price, Model, Subcategory, Category from dbo.product;
--insert into default.date(DateKey, Date, Fiscal_Quarter, Fiscal_Year, Month, Full_Date, MonthKey) select DateKey, Date, Fiscal_Quarter, Fiscal_Year, Month, Full_Date, MonthKey from dbo.date;
--insert into default.customer(CustomerKey, Customer_ID, Customer, City, State_Province, Country_Region, Postal_Code) select CustomerKey, Customer_ID, Customer, City, State_Province, Country_Region, Postal_Code from dbo.customer;

num_affected_rows,num_inserted_rows
11,11
